In [ ]:
import condor
import tqdm

import os

import numpy as np
import mrcfile

import scipy
import scipy.fft as fft

import scipy.constants as constants
from fractions import Fraction

from skimage.measure import block_reduce

import matplotlib.pyplot as plt
import matplotlib.colors

from matplotlib.colors import LogNorm
from helper_functions import (add_water_saxs, radial, electron_density_to_dn, write_text)

import sys
from sys import stderr
from datetime import date
from IPython.display import clear_output
import time

import h5py

# Elementary constants
pi = constants.pi
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

In [ ]:
phot_eV = 9000 # beam energy in eV
phot_J = phot_eV * e # beam energy in J
phot_m = (h * c) / phot_J # beam wavelength in m 
pulse_energy = 200e-6 # pulse energy in J

beam_fluence_exp = 10000e-6 # in J/um^2
focus_rad = np.sqrt(pulse_energy / (beam_fluence_exp * pi)) * 1e-6
focus_diam = 2 * focus_rad

dsf = 4

dimX = 111
dimY = 111
dimZ = 111
det_mask_ds = np.zeros(shape=(dimY,dimX))
pat = np.ones_like(det_mask_ds)
pixel_size = 800e-6
det_dist = 0.5

In [ ]:
# Write mrc file with single non-zero value
debug_grid = np.zeros(shape=(dimY,dimX,dimZ))
debug_grid[dimX//2-10:dimX//2+10,dimY//2-10:dimX//2+10,dimZ//2-10:dimX//2+10] = 5.0

with mrcfile.new('debug_grid.mrc',overwrite=True) as handle:
    handle.set_data(debug_grid.astype(np.float32))

In [ ]:
water_bg = False
save_water_bg = False
if water_bg:
    water_bg = add_water_saxs(pat, pixel_size, det_dist, phot_m, pulse_energy)
    plt.figure(dpi=90)
    plt.imshow(water_bg)
    plt.colorbar();
    sim_start, sim_end, sim_c = 0, 1, 1
    n_sim = 1000
    pat_ext = f'{n_sim}'
    water_stacked = np.broadcast_to(water_bg, (n_sim,) + water_bg.shape).astype(np.float64)
    
    for s in range(sim_start, sim_end):
        print(f"\rSimulating round {sim_c}/{sim_end-sim_start}...", flush=True)
        print(f"Simulating {n_sim} water background patterns...", flush=True)
    
        def_rng = np.random.default_rng()
        bg_water_poiss = def_rng.poisson(lam=water_stacked)

        if save_water_bg:
            time_now = time.localtime(time.time())
            base_dir = f"sims_water_only/"
            folder_name = f"run_{s}_water_{pat_ext}_pats_dsf_{dsf}x_v_2/"
            if os.path.exists(base_dir + folder_name):
                print("Path exists!", flush=True)
            else:
                print("Path does not exist...", flush=True)
                print("Making new directory...", flush=True)
                os.mkdir(base_dir + folder_name)
                print("Written!", flush=True)
            print("Writing simulation parameters...", flush=True)
            with open(base_dir + folder_name + "simulation_parameters.txt", "w") as handle:
                handle.write(f"#########################################################################################\n")
                handle.write(f"Date: {time_now[0]}/{time_now[1]}/{time_now[2]} {time_now[3]}:{time_now[4]}:{time_now[5]}\n")
                handle.write(f"Photon energy: {phot_eV} eV\n")
                handle.write(f"Downsampling factor: {dsf}\n")
                handle.write(f"Pulse energy: {pulse_energy} J\n")
                handle.write(f"Pixel size: {pixel_size} m\n")
                handle.write(f"Detector distance: {det_dist} m\n")
                handle.write(f"#########################################################################################\n")
            print("Finishing writing simulation parameters...", flush=True)
            print("Saving arrays...", flush=True)
            np.save(base_dir + folder_name + "water_only.npy",arr=water_stacked,)
            np.save(base_dir + folder_name + "poisson_water_only.npy",arr=bg_water_poiss,)
            print("Finished writing arrays...", flush=True)
            stderr.flush()
        sim_c += 1
    bg_water_poiss = bg_water_poiss.sum(axis=0)
    rad_poiss, _ = radial(bg_water_poiss)
    rad_poiss /= n_sim
    plt.figure(dpi=90)
    plt.plot(rad_poiss);

In [ ]:
with mrcfile.open('debug_grid.mrc') as handle:
    data = handle.data

plt.imshow(data[dimX//2])
plt.colorbar();

In [ ]:
source = condor.Source(wavelength=phot_m, pulse_energy=pulse_energy, focus_diameter=focus_diam, polarization='ignore',profile_model=None)

map3d, dx = condor.utils.emdio.read_map('debug_grid.mrc')
map3d_scaled = electron_density_to_dn(map3d, phot_m)
part_map = condor.ParticleMap(geometry='custom', diameter=1.0, map3d=map3d_scaled, rotation_formalism=None, dx=dx)
particle_set = {'particle_map' : part_map}

detector = condor.Detector(distance=det_dist, pixel_size=pixel_size, nx=dimX, ny=dimY)
condor_experiment = condor.Experiment(source, particle_set, detector)

sim_start, sim_end, sim_c = 0, 1, 1
n_sim = 3

for s in range(sim_start,sim_end):
    print(f'Simulating round {sim_c}/{sim_end-sim_start}...')
    print(f'Simulating {n_sim} diffraction patterns...')
    particle_intens = np.zeros(shape=(n_sim, dimY, dimX))
    
    # Resetting the RNG every simulation
    def_rng = np.random.default_rng()

    time_now = time.localtime(time.time())
    for i in tqdm.tqdm(np.arange(n_sim), colour='green'):
        result = condor_experiment.propagate()
        data_ampl = result['entry_1']['data_1']['data_fourier']
        I = np.abs(data_ampl) ** 2
        particle_intens[i, :, :] = I
write_text('All simulations done...')

In [ ]:
plt.imshow(particle_intens[0])
plt.colorbar();

In [ ]:
np.unique(particle_intens)